# Prática 2 - VITA: IA Industrial e MLOps em Seis Camadas

**Curso de IA, Otimização e IoT para Indústrias e Negócios - Do Zero ao MLOps**

Neste case, construiremos um sistema para estimar o risco de falha de bombas e motores nos próximos sete dias. O objetivo não é apenas obter uma métrica: queremos conectar sensores, transformação, ativo físico, modelo, decisão e evolução contínua de forma rastreável.

## Arquitetura do case

| Camada | Papel na prática |
| --- | --- |
| Connection | Aquisição e validação das leituras de sensores |
| Conversion | Engenharia de features e preparação dos dados |
| Ciberfísica | Estado digital, algoritmos, treinamento, inferência e registry |
| Cognição | Avaliação, métricas, diagnóstico e interpretação |
| Configuração | API, priorização e recomendação operacional |
| Consciência | Drift, governança, conhecimento, auditoria e evolução |

Tracking, versionamento e documentação conectam todas as camadas.

## Preparação

Execute este notebook a partir da raiz do projeto VITA. Se necessário, instale as dependências com `%pip install -r requirements.txt`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import os
import sys
from dotenv import load_dotenv
load_dotenv("../.env")

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT")).resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.config import DATASHEET_PATH, MODEL_CARD_PATH
from backend.layers.consciousness.governance.documents import generate_governance_documents
from backend.layers.connection.data import generate_sensor_snapshot, validate_sensor_data
from backend.layers.conversion.features import build_features, feature_definitions
from backend.layers.cyber_physical.assets import build_asset_state
from backend.layers.cyber_physical.training import train_and_register, list_runs
from backend.layers.cyber_physical.registry import load_registry, production_model
from backend.layers.cyber_physical.inference import predict_failure
from backend.layers.configuration.decision import configure_action
from backend.layers.consciousness.monitoring import monitor_drift

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'NoneType'

## 1. Connection - dados dos sensores

A primeira camada conecta o sistema às fontes de dados. Nesta prática, simularemos leituras agregadas de temperatura, vibração, corrente, carga e manutenção. Em uma aplicação real, elas poderiam chegar por gateway IoT, historiador industrial ou API.

In [ ]:
sensores = generate_sensor_snapshot(n_samples=1200, seed=42)
relatorio_validacao = validate_sensor_data(sensores)
print(relatorio_validacao)
sensores.head()

In [ ]:
figura, eixos = plt.subplots(1, 3, figsize=(16, 4))
sensores['temp_mean_24h'].hist(ax=eixos[0], bins=25)
sensores['vibration_rms_24h'].hist(ax=eixos[1], bins=25)
sensores['load_mean_24h'].hist(ax=eixos[2], bins=25)
eixos[0].set_title('Temperatura')
eixos[1].set_title('Vibração')
eixos[2].set_title('Carga')
plt.tight_layout()

## 2. Conversion - dados em informação utilizável

A camada Conversion aplica a mesma transformação no treino e na inferência. As definições recebem uma versão para evitar que produção calcule uma feature de forma diferente do experimento.

In [ ]:
features = build_features(sensores)
print(json.dumps(feature_definitions(), indent=2, ensure_ascii=False))
features[['temp_load_interaction', 'vibration_per_age']].head()

## 3. Ciberfísica - estado digital, algoritmos e modelos

Uma linha da tabela deixa de ser apenas um conjunto de números e passa a representar uma máquina física, com identidade, planta, sensores e alertas operacionais.

In [ ]:
leitura_exemplo = {
    'machine_id': 42,
    'plant': 'SP',
    'motor_age_days': 1200,
    'temp_mean_24h': 88.5,
    'vibration_rms_24h': 5.2,
    'current_mean_24h': 19.5,
    'load_mean_24h': 0.97,
    'maintenance_last_30d': 0,
}
estado_ativo = build_asset_state(leitura_exemplo)
estado_ativo

## 4. Cognition - tracking de experimentos

Treinaremos três candidatos. Cada execução registra algoritmo, seed, limiar, métricas, hash dos dados, versão das features, commit e artefato. O MLflow usa um banco SQLite local; uma tabela CSV permanece disponível para inspeção direta.

In [ ]:
resultado_treino = train_and_register(n_samples=1200, seed=42)
runs = pd.DataFrame(resultado_treino['runs'])
runs[['run_id', 'algorithm', 'roc_auc', 'f1', 'recall_failure', 'expected_cost']]

Para abrir a interface do MLflow em outro terminal:

```bash
mlflow ui --backend-store-uri sqlite:///artifacts/tracking/mlflow.db --port 5000
```

Acesse `http://localhost:5000`.

## 5. Registry - seleção, aprovação e promoção

O melhor modelo não é escolhido apenas pela maior acurácia. Neste case, falsos negativos têm custo maior. O candidato com menor custo esperado e bom recall é promovido para produção; versões anteriores são arquivadas.

In [ ]:
registry = load_registry()
print('Versão em produção:', registry['production_version'])
pd.DataFrame(registry['models'])[['version', 'algorithm', 'stage', 'source_run_id', 'code_commit']]

## 6. Versionamento e rastreabilidade

O manifesto registra a combinação que produziu o resultado: dados, features, código, dependências, configuração e seed. O hash permite detectar alterações mesmo quando o nome do arquivo não muda.

In [ ]:
manifesto = resultado_treino['manifest']
print(json.dumps(manifesto, indent=2, ensure_ascii=False))

## 7. Cognition - inferência com a versão aprovada

In [ ]:
predicao = predict_failure(leitura_exemplo)
predicao

## 8. Configuration - da probabilidade à decisão

O modelo estima risco; a camada Configuration combina essa estimativa com o estado físico e produz uma recomendação. Neste exemplo, nenhuma decisão desliga automaticamente o equipamento.

In [ ]:
decisao = configure_action(
    predicao['risk_probability'],
    predicao['threshold'],
    estado_ativo,
)
decisao

## 9. Consciência - monitoramento, governança e retroalimentação

Simularemos uma mudança de temperatura, vibração e carga. O PSI compara a distribuição atual com a referência de treino. Um alerta não prova que o modelo falhou; ele exige investigação.

In [ ]:
relatorio_drift = monitor_drift(n_samples=1000, seed=99, drift=True)
drift_df = pd.DataFrame.from_dict(relatorio_drift['features'], orient='index')
drift_df

In [ ]:
cores = ['#b91c1c' if status == 'alert' else '#15803d' for status in drift_df['status']]
drift_df['psi'].plot(kind='bar', color=cores, figsize=(12, 4), title='PSI por feature')
plt.axhline(0.2, color='orange', linestyle='--', label='limiar de investigação')
plt.legend()
plt.tight_layout()

## 10. Transparência - model card e datasheet

A documentação é gerada a partir das evidências da versão aprovada. Ela registra finalidade, métricas, limitações, usos proibidos, origem dos dados e política de monitoramento.

In [ ]:
documentos = generate_governance_documents(
    resultado_treino['selected_model'],
    resultado_treino['manifest'],
)
documentos

In [ ]:
print(MODEL_CARD_PATH.read_text(encoding='utf-8'))

## 11. Auditoria - reconstruindo a história

A pergunta final é: se uma previsão gerar uma decisão incorreta, conseguimos reconstruir qual modelo, run, dataset, feature set, código e limiar foram usados?

In [ ]:
modelo_producao = production_model()
auditoria = {
    'modelo': modelo_producao['registered_model_name'],
    'versao': modelo_producao['version'],
    'run_id': modelo_producao['source_run_id'],
    'dataset': modelo_producao['data_version']['name'],
    'dataset_sha256': modelo_producao['data_version']['sha256'],
    'feature_set': modelo_producao['feature_set_version'],
    'code_commit': modelo_producao['code_commit'],
    'model_sha256': modelo_producao['model_sha256'],
    'limiar': modelo_producao['decision_threshold'],
    'model_card': str(MODEL_CARD_PATH),
    'datasheet': str(DATASHEET_PATH),
}
print(json.dumps(auditoria, indent=2, ensure_ascii=False))

## Atividade

1. Altere a seed e compare os novos runs.
2. Explique por que o algoritmo com maior F1 pode não ser o escolhido.
3. Identifique as evidências necessárias para reproduzir a versão em produção.
4. Execute o monitoramento sem drift e compare os relatórios.
5. Revise o model card e acrescente uma limitação operacional.
6. Explique por que o sistema não executa desligamento automático.

## Conclusão

O valor do case não está apenas no classificador. Ele está na conexão entre dados confiáveis, estado do ativo, modelo rastreável, decisão responsável e aprendizado contínuo. Essa conexão materializa a arquitetura VITA e transforma um experimento em um sistema industrial auditável.